In [1]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt


In [2]:
import torch
print(torch.__version__)

2.6.0+cu124


In [2]:

# 定义神经网络模型类
class Net(torch.nn.Module):
    # 初始化方法
    def __init__(self):
        # 继承父类torch.nn.Module的初始化
        super().__init__()
        # 第一层全连接层：输入维度28*28（784），输出维度64
        self.fc1 = torch.nn.Linear(28*28, 64)
        # 第二层全连接层：输入维度64，输出维度64
        self.fc2 = torch.nn.Linear(64, 64)
        # 第三层全连接层：输入维度64，输出维度64
        self.fc3 = torch.nn.Linear(64, 64)
        # 输出层全连接层：输入维度64，输出维度10（0-9共10个数字类别）
        self.fc4 = torch.nn.Linear(64, 10)
    
    # 前向传播方法
    def forward(self, x):
        # 将输入x通过fc1，使用ReLU激活函数
        x = torch.nn.functional.relu(self.fc1(x))
        # 将上一层输出通过fc2，使用ReLU激活函数
        x = torch.nn.functional.relu(self.fc2(x))
        # 将上一层输出通过fc3，使用ReLU激活函数
        x = torch.nn.functional.relu(self.fc3(x))
        # 将上一层输出通过fc4，使用log_softmax激活函数（用于多分类）
        x = torch.nn.functional.log_softmax(self.fc4(x), dim=1)
        # 返回最终的输出
        return x


# 获取数据加载器函数
def get_data_loader(is_train):
    # 定义数据转换为张量的操作
    to_tensor = transforms.Compose([transforms.ToTensor()])
    # 加载MNIST数据集，is_train判断是训练集还是测试集，自动下载数据
    data_set = MNIST("", is_train, transform=to_tensor, download=True)
    # 创建数据加载器，批次大小为15，随机打乱数据
    return DataLoader(data_set, batch_size=15, shuffle=True)


# 模型评估函数
def evaluate(test_data, net):
    # 初始化正确预测的数量
    n_correct = 0
    # 初始化总预测数量
    n_total = 0
    # 禁用梯度计算（评估时不需要反向传播）
    with torch.no_grad():
        # 遍历测试数据加载器中的每个批次
        for (x, y) in test_data:
            # 将图像reshape为1维向量（-1自动推断为批次大小，28*28为像素总数）
            outputs = net.forward(x.view(-1, 28*28))
            # 遍历当前批次的每个样本的输出
            for i, output in enumerate(outputs):
                # 找到输出中最大值的索引作为预测类别，与真实标签y[i]比较
                if torch.argmax(output) == y[i]:
                    # 如果预测正确，正确数加1
                    n_correct += 1
                # 总数加1
                n_total += 1
    # 返回准确率（正确预测数/总预测数）
    return n_correct / n_total


# 主函数
def main():
    # 加载训练数据集
    train_data = get_data_loader(is_train=True)
    # 加载测试数据集
    test_data = get_data_loader(is_train=False)
    # 创建神经网络模型实例
    net = Net()
    
    # 打印初始（训练前）模型的准确率
    print("initial accuracy:", evaluate(test_data, net))
    # 创建Adam优化器，学习率为0.001
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
    # 循环训练2个epoch
    for epoch in range(2):
        # 遍历训练数据加载器中的每个批次
        for (x, y) in train_data:
            # 清空优化器中累积的梯度
            net.zero_grad()
            # 前向传播：将图像reshape为1维向量，传入网络得到输出
            output = net.forward(x.view(-1, 28*28))
            # 计算负对数损失函数（用于多分类任务）
            loss = torch.nn.functional.nll_loss(output, y)
            # 反向传播计算梯度
            loss.backward()
            # 使用优化器更新模型参数
            optimizer.step()
        # 每个epoch后打印当前准确率
        print("epoch", epoch, "accuracy:", evaluate(test_data, net))

    # 遍历测试数据集，最多显示4个样本
    for (n, (x, _)) in enumerate(test_data):
        # 只显示前4个批次的第一个样本
        if n > 3:
            break
        # 预测第一个样本的类别（返回概率最大的类别索引）
        predict = torch.argmax(net.forward(x[0].view(-1, 28*28)))
        # 创建新的图形窗口
        plt.figure(n)
        # 显示图像（28x28像素）
        plt.imshow(x[0].view(28, 28))
        # 设置图像标题为预测的数字
        plt.title("prediction: " + str(int(predict)))
    # 显示所有图形
    plt.show()


# 程序入口点：如果这个脚本被直接运行（不是被导入），执行main函数
if __name__ == "__main__":
    main()


100.0%
100.0%
100.0%
100.0%


initial accuracy: 0.098
epoch 0 accuracy: 0.9606
epoch 1 accuracy: 0.9661


: 